# microsoft_opentelemetry smoke test

Microsoft's OpenTelemetry distribution bundling otel-api/sdk/instrumentation pins + Azure Monitor exporter.

In [ ]:
# Example 1: Inspect the public API surface of microsoft_opentelemetry.
import microsoft.opentelemetry as microsoft_opentelemetry  # dist installs the microsoft.opentelemetry namespace

public_attrs = [a for a in dir(microsoft_opentelemetry) if not a.startswith('_')]
print(f"Public attributes count: {len(public_attrs)}")
print(f"Sample: {public_attrs[:15]}")

# Show types of public attributes
for attr_name in public_attrs[:10]:
    attr = getattr(microsoft_opentelemetry, attr_name)
    print(f"  {attr_name}: {type(attr).__name__}")

assert microsoft_opentelemetry.__path__, "package __path__ should be non-empty"

In [ ]:
# Example 2: Verify core OpenTelemetry API imports work via the bundled distribution.
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(ConsoleSpanExporter()))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("microsoft_opentelemetry.test")
with tracer.start_as_current_span("smoke-span") as span:
    span.set_attribute("test.attr", "value")
    current_span = trace.get_current_span()
    assert current_span.is_recording(), "span should be recording"

print("Span created and recorded")

In [ ]:
# Example 3: The bundled Azure Monitor exporters import, and a trace exporter can be built offline.
from azure.monitor.opentelemetry.exporter import (
    AzureMonitorTraceExporter,
    AzureMonitorLogExporter,
    AzureMonitorMetricExporter,
)
from opentelemetry.sdk.trace.export import SpanExporter

# Construction only parses the connection string; nothing is exported. The all-zero key is a dummy, and
# disable_offline_storage stops the exporter from creating its retry folder on disk.
trace_exporter = AzureMonitorTraceExporter(
    connection_string="InstrumentationKey=00000000-0000-0000-0000-000000000000",
    disable_offline_storage=True,
)
assert isinstance(trace_exporter, AzureMonitorTraceExporter) and isinstance(trace_exporter, SpanExporter)
trace_exporter.shutdown()

# Without a connection string the exporter refuses to start.
try:
    AzureMonitorTraceExporter()
except ValueError as missing_key:
    print(f"no connection string -> ValueError: {missing_key}")
else:
    raise AssertionError("AzureMonitorTraceExporter() accepted a missing connection string")

print(f"AzureMonitorTraceExporter: {AzureMonitorTraceExporter}")
print(f"AzureMonitorLogExporter: {AzureMonitorLogExporter}")
print(f"AzureMonitorMetricExporter: {AzureMonitorMetricExporter}")

In [ ]:
# Example 4: Verify a sampling of bundled instrumentation packages import.
instrumentations = [
    'opentelemetry.instrumentation.requests',
    'opentelemetry.instrumentation.urllib',
    'opentelemetry.instrumentation.urllib3',
    'opentelemetry.instrumentation.httpx',
    'opentelemetry.instrumentation.logging',
]

import importlib
loaded = []
for mod_name in instrumentations:
    mod = importlib.import_module(mod_name)
    loaded.append(mod_name)
    print(f"  {mod_name}: ok")

assert len(loaded) == len(instrumentations), "all instrumentations should import"

In [ ]:
# Example 5: Exercise the OpenTelemetry resource detection / resource API end-to-end.
from opentelemetry.sdk.resources import Resource, SERVICE_NAME

resource = Resource.create({
    SERVICE_NAME: "microsoft-opentelemetry-smoke-test",
    "service.version": "0.0.1",
    "deployment.environment": "test",
})

attrs = dict(resource.attributes)
print(f"Resource attributes: {attrs}")

assert attrs[SERVICE_NAME] == "microsoft-opentelemetry-smoke-test"
assert attrs["service.version"] == "0.0.1"
assert attrs["deployment.environment"] == "test"
print("Resource API works as expected")